# Bài tập Buổi 5 — Pipeline Machine Learning: EDA & Tiền xử lý trên Titanic

**Khóa học hè 2026 — Python & Machine Learning · ML IoT Lab, HCMUT**

---

## Bối cảnh

Bạn vừa nhận được dataset **Titanic** — danh sách hành khách và việc họ có sống sót sau thảm họa hay không.
Nhiệm vụ của bạn **không phải** huấn luyện mô hình, mà là hoàn thành **hai bước đầu và quan trọng nhất** của một dự án Machine Learning:

> **Khám phá dữ liệu (EDA) → Tiền xử lý dữ liệu.**

Đây là phần chiếm ~70–90% công sức thực tế của một dự án ML. Một mô hình mạnh không thể cứu một bộ dữ liệu kém chất lượng.

## Mục tiêu bài tập

Sau khi hoàn thành, bạn sẽ chứng minh được rằng mình có thể:

1. Thực hiện **EDA đầy đủ** trên một dataset thực tế: kiểm tra cấu trúc, giá trị thiếu, outlier, phân phối và tương quan.
2. **Trực quan hóa** dữ liệu và **rút ra nhận xét có căn cứ** (không chỉ vẽ hình cho đẹp).
3. Áp dụng **đúng kỹ thuật tiền xử lý** cho từng loại dữ liệu: xử lý missing, encoding, scaling.
4. Chia tập và xây pipeline tiền xử lý **không rò rỉ dữ liệu (data leakage)**.
5. Viết **nhận xét tổng hợp** về dữ liệu như một data analyst thực thụ.

## Yêu cầu nộp bài

- Hoàn thiện notebook này (điền vào tất cả các ô `# TODO` và các phần *"Trả lời:"*).
- Notebook phải **chạy được từ trên xuống dưới không lỗi** (Kernel → Restart & Run All).
- Nộp qua **GitHub**: tải repo mẫu → đưa lên repo cá nhân → làm bài và nộp trên đó.

## Tiêu chí chấm (10 điểm)

| Nội dung | Điểm |
|---|---|
| EDA đầy đủ (shape/info/missing/outlier) | 2.0 |
| Trực quan hóa + nhận xét cho mỗi biểu đồ | 2.0 |
| Xử lý missing & outlier hợp lý, có giải thích | 1.5 |
| Encoding & scaling đúng loại biến | 1.5 |
| Chia tập & tiền xử lý **không leakage** | 1.5 |
| Nhận xét tổng hợp về dữ liệu | 1.5 |

> **Lưu ý về liêm chính học thuật:** được tham khảo tài liệu, nhưng phải **tự viết code và tự hiểu**. Phần nhận xét phải là quan sát của chính bạn từ dữ liệu.

---


## 0. Chuẩn bị môi trường

Ô này đã viết sẵn — chạy để nạp thư viện. Nếu thiếu thư viện, cài bằng `pip install pandas numpy matplotlib seaborn scikit-learn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)          # cố định ngẫu nhiên -> kết quả tái lập được
print("Sẵn sàng.")

## 1. Tải dữ liệu (đã cho)

Ô này đã viết sẵn. Dữ liệu được tải từ `seaborn`, có fallback tải từ Internet nếu cần.

In [ ]:
try:
    df = sns.load_dataset("titanic")
    print("Đã tải từ seaborn.")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    df.columns = [c.lower() for c in df.columns]
    print("Đã tải từ URL.")
df.head()

---
## Task 1 — Loại bỏ cột rò rỉ nhãn (data leakage) và cột dư thừa

### Mục đích
Trên slide đã học: **data leakage** là khi thông tin không được phép "rò" vào mô hình, khiến kết quả đẹp trên giấy nhưng vô dụng thực tế. Bản Titanic của seaborn chứa sẵn nhiều cột **rò rỉ nhãn** hoặc **trùng lặp**:

- `alive` (yes/no) — chính là `survived` viết bằng chữ ⇒ **rò rỉ target trực tiếp**. Để lại là mô hình "gian lận".
- `who`, `adult_male`, `class` — được suy ra từ `sex`, `age`, `pclass` (trùng thông tin).
- `deck` — thiếu quá nhiều (~77%).
- `embark_town` — trùng `embarked`; `alone` — suy ra từ `sibsp` + `parch`.

### Yêu cầu
1. In ra danh sách cột và tỷ lệ missing của **toàn bộ** dataframe (để thấy `deck` thiếu bao nhiêu).
2. Loại bỏ các cột rò rỉ / dư thừa ở trên, chỉ giữ lại:
   `survived, pclass, sex, age, sibsp, parch, fare, embarked`.
3. **Trả lời** (markdown ô dưới): vì sao để lại cột `alive` sẽ khiến mô hình đạt accuracy ~100% mà không thực sự học được gì?

### Gợi ý
- `df.isnull().mean()` cho tỷ lệ thiếu theo cột.
- `df.drop(columns=[...])` để bỏ cột.

In [ ]:
# TODO 1a: in tỷ lệ missing của tất cả các cột

missing_ratio = df.isnull().mean() * 100
print("Tỷ lệ missing của tất cả các cột:")
print(missing_ratio.sort_values(ascending=False))

# Hàm isnull sẽ kiểm tra ô nào bị thiếu dữ liệu: Có dữ liệu -> False, Thiếu dữ liệu (NaN) -> True
# Hàm mean tính trung bình

# TODO 1b: bỏ các cột rò rỉ/dư thừa, gán lại vào biến df
leaky = [
    "alive", # rò rỉ target
    "who", # suy ra từ sex và age
    "adult_male", # suy ra từ sex và age
    "class", # trùng với pclass
    "deck", # thiếu dữ liệu quá nhiều
    "embark_town", # trùng với embarked
    "alone" # suy ra từ sibsp và parch
]      # điền danh sách cột cần bỏ (chỉ những cột có trong df)

# Chỉ xóa những cột tồn tại trong dataFrame
leaky = [col for col in leaky if col in df.columns]

# df = df.drop(columns=...)
df = df.drop(columns=leaky)

# print("Các cột còn lại:", list(df.columns))
print("Các cột còn lại:")
print(list(df.columns))

**Trả lời 1c (vì sao `alive` gây rò rỉ target):**

alive đơn giản là cách biểu diễn khác của biến survived. survived = 1 thì alive ='yes'
survived =0 thì alive = 'no'
Khi có cột alive, mô hình chạy chỉ ccho ra "yes -> 1, no -> 0" mà không đề cập đến những yếu tố quan trọng như tuổi, giới tính,... Điều này làm cho mô hình có Accuracy đạt giá trị cao, thiếu sự nhất quán và sai sót khá nhiều. Trên thực tế cũng có thể thấy, khi dự đoán hành khách mới thì alive không cần thiết nên không tồn tại.

---
## Task 2 — Quan sát tổng quan

### Mục đích
Trước khi phân tích sâu, phải nắm được "hình dạng" của dữ liệu: bao nhiêu mẫu, bao nhiêu đặc trưng, kiểu dữ liệu từng cột, và thống kê cơ bản. Đây là bước đầu tiên của mọi EDA.

### Yêu cầu
1. In **số dòng và số cột**; nêu rõ đâu là **biến mục tiêu (target)**.
2. Dùng `df.info()` để xem kiểu dữ liệu và số giá trị non-null.
3. Dùng `df.describe()` cho biến số và `df.describe(include="object")` (hoặc `"category"`) cho biến phân loại.
4. **Trả lời:** cột nào là biến **số**, cột nào là biến **phân loại**?

In [ ]:
# TODO 2: shape, info, describe
print(f"Số dòng: {df.shape[0]}") # số dòng
print(f"Số cột: {df.shape[1]}") # số cột
print("Biến mục tiêu (target): survived")

print("Thông tin dataframe:")
df.info()

print("Thống kê các biến số:")
print(df.describe())

print("Thống kế các biến phân loại:")
print(df.describe(include="object"))

**Trả lời 2 (biến số vs biến phân loại):**
Biến số: survived, pclass, age, sibsp, parch, fare.
Biến phân loại: sex, embarked.
survived và pclass trên tệp được lưu dưới dạng số nhưng chúng biểu diễn các nhóm (0/1 và hạng vé 1/2/3) nên khi xây dựng mô hình có thể xem đây là các biến phân loại được mã hóa bằng số.

---
## Task 3 — Missing Value: thống kê & đề xuất cách xử lý

### Mục đích
Mô hình học máy **không nhận trực tiếp giá trị NaN**. Nhưng cách xử lý phụ thuộc **tỷ lệ thiếu** và **vai trò của cột** — không có một cách đúng cho mọi trường hợp.

### Yêu cầu
1. Lập bảng: mỗi cột còn missing → **số lượng** và **phần trăm** thiếu.
2. Với **từng cột** còn thiếu, **đề xuất** cách xử lý và **giải thích ngắn gọn** (xóa / điền mean / điền median / điền mode / KNN...).

### Gợi ý
- Nhắc lại từ slide: `median` bền vững hơn `mean` khi có outlier; cột thiếu quá nhiều (>~60–70%) thường nên **bỏ**; biến phân loại thường điền **mode**.

In [ ]:
# TODO 3: bảng missing (count + %)
missing_table = pd.DataFrame({
    "Số giá trị bị thiếu của từng cột": df.isnull().sum(),
    "Tỷ lệ thiếu (%)": df.isnull().mean() * 100
})

# Chỉ hiển thị các cột còn thiếu dữ liệu
missing_table = missing_table[missing_table["Số giá trị bị thiếu của từng cột"] > 0]
print(missing_table)

**Trả lời 3 (đề xuất xử lý cho từng cột thiếu):**

| Cột | % thiếu | Cách xử lý đề xuất | Lý do |
|age|gần bằng 19.87%|Điền bằng median|age là biến số và có thể có ngoại lai (outlier). Chọn median vì median ít bị ảnh hưởng bởi outlier hơn mean.|
| embarked | gần bằng 0.22% | Điền bằng mode | embarked là biến phân loại và thiếu rất ít dữ liệu, nên sử dụng mode để cho ra giá trị xuất hiện nhiều nhất |

---
## Task 4 — Phát hiện Outlier & **ra quyết định**

### Mục đích
Outlier có thể là **lỗi nhập liệu** (cần xử lý) hoặc **hiện tượng thật** (cần giữ). Phát hiện thôi chưa đủ — một analyst giỏi phải **quyết định** làm gì và giải thích được.

### Yêu cầu
1. Trên hai cột `age` và `fare`, đếm số outlier bằng **cả hai** phương pháp: **IQR** và **Z-score** (ngưỡng |z| > 3).
2. **Trả lời:** với các outlier của `fare`, bạn **giữ lại hay loại bỏ**? Vì sao? (gợi ý: nghĩ xem vé đắt bất thường là lỗi hay là vé hạng nhất có thật).

### Gợi ý
- IQR: outlier là điểm ngoài khoảng `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`.
- Z-score: `from scipy import stats; np.abs(stats.zscore(series.dropna()))`.

In [ ]:
# TODO 4: đếm outlier theo IQR và Z-score cho 'age' và 'fare'
def dem_outlier_iqr(s):
    s = s.dropna() # không thể tính IQR nếu còn giá trị thiếu nên bỏ NaN
    
    Q1 = s.quantile(0.25)
    Q3 = s.quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    return ((s < lower) | (s > upper)).sum()    # trả về số lượng outlier theo IQR

def dem_outlier_zscore(s, nguong=3.0):
    s = s.dropna()
    
    z = np.abs(stats.zscore(s))
    
    return (z > nguong).sum()    # trả về số lượng outlier theo Z-score

# for col in ["age", "fare"]:
for col in ["age", "fare"]:
    print(f"cột: {col}")
    print("Số outlier theo IQR:", dem_outlier_iqr(df[col]))
    print("Số outlier theo Z-score:", dem_outlier_zscore(df[col]))

**Trả lời 4 (quyết định với outlier của `fare`):**
Outlier của `fare` nên được giữ lại. Mức giá vé cao có thể xem là hành khách đi hạng nhất hay mua những loại vé đặc biệt khác. Nếu loại bỏ các giá trị này có thể làm mất những thông tin quan trọng về sự ảnh hưởng của giá vé lên khả năng sống sót của hành khách.

---
## Task 5 — Trực quan hóa & nhận xét

### Mục đích
EDA là môn học về **nhìn** dữ liệu. Mỗi biểu đồ phải trả lời một câu hỏi và **đi kèm một nhận xét**. Vẽ mà không nhận xét thì không tính điểm.

### Yêu cầu — vẽ tối thiểu 4 loại biểu đồ, mỗi biểu đồ 1–2 câu nhận xét:
1. **Univariate — Histogram**: phân phối của `age` và `fare`. (Nhận xét: có lệch không? lệch trái hay phải?)
2. **Univariate — Boxplot**: `fare` theo nhóm `survived` hoặc `pclass`. (Nhận xét: outlier, trung vị.)
3. **Bivariate — Bar/Barplot**: **tỷ lệ sống sót** theo `sex` và theo `pclass`. (Nhận xét: nhóm nào sống nhiều hơn, chênh bao nhiêu %?)
4. **Multivariate — Heatmap**: ma trận tương quan giữa các biến số. (Nhận xét: cặp biến nào tương quan mạnh?)

### Gợi ý
- `sns.histplot`, `sns.boxplot`, `sns.barplot(data=df, x="sex", y="survived")` (barplot tự tính trung bình = tỷ lệ sống sót), `sns.heatmap(df.select_dtypes("number").corr(), annot=True)`.

In [ ]:
# TODO 5a: Histogram age & fare
# Histogram của age
plt.figure(figsize=(6,4))
sns.histplot(df["age"], bins=20, kde=True)
plt.title("Histogram của Age")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

# Histogram của fare
plt.figure(figsize=(6,4))
sns.histplot(df["fare"], bins=20, kde=True)
plt.title("Histogram của Fare")
plt.xlabel("Fare")
plt.ylabel("Count")
plt.show()

In [ ]:
# TODO 5b: Boxplot fare theo survived hoặc pclass
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="pclass", y="fare")
plt.title("Boxplot fare theo pclass")
plt.xlabel("Pclass")
plt.ylabel("Fare")
plt.show()

In [ ]:
# TODO 5c: Barplot tỷ lệ sống sót theo sex và pclass
# Barplot tỷ lệ sống sót theo sex
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="sex", y="survived")
plt.title("Barplot tỷ lệ sống sót theo Sex")
plt.xlabel("Sex")
plt.ylabel("Tỉ lệ sống sót")
plt.show()

# Barplot tỷ lệ sống sót theo pclass
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x="pclass", y="survived")
plt.title("Barplot tỷ lệ sống sót theo Pclass")
plt.xlabel("Pclass")
plt.ylabel("Tỉ lệ sống sót")
plt.show()

In [ ]:
# TODO 5d: Heatmap correlation
plt.figure(figsize=(6,4))
corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Heatmap correlation")
plt.show()

**Nhận xét 5 (viết cho từng biểu đồ ở trên):**

- Histogram: Age tập trung chủ yếu khoảng độ tuổi từ 20-40 và có phân phối lệch phải. Fare lệch hẳn sang phải chứng tỏ đa số hành khách mua vé giá thấp và thiểu số mua vé với giá rất cao.
- Boxplot: có nhiều outlieer ở mức giá cao. Hành khách hạng 1 có giá vé cao nhất, hành khách hạng 3 có giá vé thấp nhất.
- Bar survival: nữ có tỷ lệ sống sót cao hơn nam. Hành khách hạng 1 và hạng 2 có tỷ lệ sống sót khá tương đồng nhau và chiếm vị trí cao nhất, hành khác hạng 3 có tỷ lệ sống sót thấp nhất trong cả 3 hạng.
- Heatmap: không có cặp biến nào tương quan quá mạnh. Cặp pclass và fare có tương quan âm mạnh nhất, chứng tỏ hành khách hạng vé cao (pclass nhỏ) thường trả giá vé cao. Cặp sibsp và parch có tương quan dương mức trung bình, chứng tỏ những người đi cùng gia đình thường có anh, chị, em,...(sibsp) và cha mẹ, con cái (parch). Đặc biệt, biến survived có tương quan dương với fare và tương quan âm với pclass cho thấy hành khách mua vé đắt hoặc hạng cao có xu hướng sống sót cao hơn.

---
## Task 6 — Chia tập **TRƯỚC** khi tiền xử lý (chống data leakage)

### Mục đích
Đây là điểm mấu chốt của buổi học. Mọi phép "học tham số" từ dữ liệu (median để điền, min/max/IQR để scale, danh mục để encode) **chỉ được học từ tập train**. Nếu học từ toàn bộ dữ liệu rồi mới chia, thông tin của tập test đã **rò rỉ** — điểm đánh giá sẽ ảo.

⇒ **Vì vậy phải chia tập TRƯỚC**, rồi mới xử lý.

### Yêu cầu
1. Tách `X` (đặc trưng) và `y` (`survived`).
2. Chia **train / validation / test** theo tỷ lệ khoảng **70 / 15 / 15**, có **`stratify=y`** để giữ nguyên tỷ lệ hai lớp.
3. In shape của 3 tập và **tỷ lệ sống sót** trong mỗi tập (để kiểm tra stratify hoạt động).

### Gợi ý
- Dùng `train_test_split` **hai lần**: lần 1 tách test (15%), lần 2 tách val từ phần còn lại.
- `stratify` nhận vào nhãn tương ứng ở mỗi lần chia.

In [ ]:
# TODO 6: chia train/val/test có stratify
X = df.drop(columns="survived") # bỏ cột survived ra khỏi dữ liệu
y = df["survived"] # lấy riêng cột survived

# X_tmp, X_test, y_tmp, y_test = train_test_split(...)
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X,
    y,
    test_size=0.15, # lấy 15% dữ liệu làm test
    random_state=10, # giữ cho việc chia dữ liệu luôn giống nhau mỗi lần chạy
    stratify=y
)
# X_train, X_val, y_train, y_val = train_test_split(...)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp,
    y_tmp,
    test_size=0.1765, # lấy 15% của 85% còn lại trước đó
    random_state=10,
    stratify=y_tmp
)

# print("Train/Val/Test:", ...)
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

# in tỷ lệ survived từng tập
print("Tỷ lệ sống sót:")
print(f"Train: {y_train.mean():.3f}")
print(f"Validation: {y_val.mean():.3f}")
print(f"Test: {y_test.mean():.3f}")

---
## Task 7 — Xây pipeline tiền xử lý, **fit chỉ trên train**

### Mục đích
Gộp toàn bộ bước tiền xử lý vào một `ColumnTransformer` + `Pipeline`, `fit` **một lần trên `X_train`** rồi `transform` cho val/test. Đây là cách chuẩn để **đảm bảo không leakage** và tái sử dụng được.

### Yêu cầu
Xây `preprocess` gồm:

- **Biến số** (`age`, `sibsp`, `parch`, `fare`): `SimpleImputer(median)` → scaler (chọn `RobustScaler` vì `fare` có outlier, hoặc giải thích lựa chọn khác).
- **Biến phân loại** (`sex`, `embarked`): `SimpleImputer(most_frequent)` → `OneHotEncoder`.
- **Biến thứ tự** (`pclass`): giữ nguyên (`passthrough`) vì đã là số có thứ tự 1 < 2 < 3.

Sau đó: `fit` trên `X_train`, `transform` cho cả ba tập; in shape kết quả và tên cột sau biến đổi.

### Yêu cầu trả lời
- **Trả lời:** giải thích vì sao `fit` chỉ trên train (không phải trên toàn bộ dữ liệu) thì tránh được leakage.

### Gợi ý
- Khung `ColumnTransformer([... ("num", pipe_so, num_cols), ("cat", pipe_cat, cat_cols), ("ord", "passthrough", ord_cols)])`.
- `preprocess.get_feature_names_out()` để xem tên cột sau biến đổi.

In [ ]:
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["pclass"]

# TODO 7: xây pipeline cho biến số và biến phân loại
pipe_so = Pipeline([
    ("imputer", SimpleImputer(strategy="median")), # thay giá trị bị thiếu bằng median
    ("scaler",  RobustScaler()), # cột fare có nhiều outlier nên dùng này ít bị ảnh hưởng hơn
])
pipe_cat = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")), # thay giá trị bị thiếu bằng mode
    ("onehot",  OneHotEncoder(handle_unknown="ignore")), # máy học không hiểu chữ nên phải đổi thành số
])

preprocess = ColumnTransformer([
    ("num", pipe_so,  num_cols),
    ("cat", pipe_cat, cat_cols),
    ("ord", "passthrough", ord_cols),
])

preprocess.fit(X_train)               # fit CHỈ trên train
X_train_t = preprocess.transform(X_train)

# ... transform cho val, test
X_val_t = preprocess.transform(X_val)
X_test_t = preprocess.transform(X_test)

# print(X_train_t.shape, list(preprocess.get_feature_names_out()))
print("Train:", X_train_t.shape)
print("Validation:", X_val_t.shape)
print("Test:", X_test_t.shape)
print(preprocess.get_feature_names_out())

**Trả lời 7 (vì sao fit chỉ trên train tránh leakage):**
fit chỉ trên tập train giúp các tham số như median, mode, scaler và OneHotEncoder chỉ được học từ dữ liệu huấn luyện. Các tham số này lần lượt được transform cho tập validation và test. Nếu fit trên toàn bộ dữ liệu thì thông tin chưa tới fit đã bị leakage trong quá trình tiền xử lý trước đó, từ đó làm mô hình không còn chính xác nữa.

---
## Task 8 — Câu hỏi tư duy: chọn metric đánh giá

### Mục đích
Buổi học nhấn mạnh: **không có metric tốt nhất tuyệt đối** — phải chọn theo bài toán và mức mất cân bằng dữ liệu. Bài này không cần code, chỉ cần lập luận.

### Yêu cầu — trả lời ngắn gọn:
1. Biến mục tiêu `survived` có **mất cân bằng** không? (tính tỷ lệ hai lớp để trả lời).
2. Nếu chỉ nhìn **Accuracy**, có thể bị đánh lừa trong trường hợp nào?
3. Với bài toán Titanic, bạn sẽ ưu tiên metric nào (Accuracy / Precision / Recall / F1)? Vì sao?

In [ ]:
# TODO 8: tính tỷ lệ hai lớp của 'survived' để hỗ trợ trả lời
survival_rate = df["survived"].value_counts(normalize=True) * 100
print("Tỷ lệ hai lớp cho biến survived:")
print(survival_rate)

**Trả lời 8:**

1. Tỷ lệ là 61.6% hành khách không sống sót và 38.4% hành khách sống sót. Hai lớp không chênh lệch quá lớn nhưng cũng không hoàn toàn cân bằng.
2. Nếu mô hình luôn dự đoán tất cả hành khách không sống sót, nó vẫn đạt 61.6% mặc dù không dự đoán đúng bất kỳ trường hợp sống sót nào.
3. Em chọn F1-score vì nó có khả năng cân bằng giữa Precision và Recall giúp đánh giá mô hình tốt hơn khi dữ liệu bị mất cân bằng. 

---
## Task 9 — Nhận xét tổng hợp về dữ liệu

### Mục đích
Khép lại toàn bộ EDA bằng một bản tóm tắt như một data analyst gửi cho đồng đội: **những gì đáng chú ý nhất** về bộ dữ liệu này.

### Yêu cầu — viết ít nhất 5 gạch đầu dòng, dựa trên **bằng chứng** (số liệu / biểu đồ) ở trên:
- Đặc trưng nào **tương quan mạnh nhất** với khả năng sống sót? (số liệu chứng minh)
- Cột nào **thiếu nhiều nhất** và bạn đã xử lý thế nào?
- Biến mục tiêu có **mất cân bằng** không? ảnh hưởng gì tới việc chọn metric?
- Đặc trưng nào cần **scaling**, đặc trưng nào cần **encoding**? vì sao?
- Một điều bạn thấy **bất ngờ / thú vị** trong dữ liệu.

**Nhận xét tổng hợp của bạn:**

1. Đặc trưng tương quan mạnh nhất với khả năng sống sót là pclass và fare. Biến survived tương quan âm với pclass (gần bằng 0.34) và tương quan dương với fare (gần bằng 0.26). 
2. Cột thiếu nhiều nhất là Age (19.9%), có outlier nên xử lý bằng median. 
3. Biến mục tiêu survived có mất cân bằng nhưng ở mức độ nhẹ, với 61.6% không sống sót và 38.4% sống sót. Vì vậy, nên sử dụng Accuracy với mục đích xem bổ sung để hiểu hơn về mô hình, còn cái chính vẫn là F1-score để đánh giá mô hình đầy đủ hơn, công bằng hơn.
4. Đặc trưng cần scaling là: age, sibsp, parch và fare. Các biến được scaling bằng RobustScaler để đưa về cùng thang đo, giúp mô hình học hiệu quả hơn. Đặc trưng cần Encoding là: sex và embarked. Các biến này cần được Encodin bằng OneHotEncoder để chuyển các giá trị chữ thành dạng số mà mô hình có thể xử lý.
5. Điều em thấy khá thú vị là nữ giới tại sao lại có tỉ lệ sống sót cao hơn nam giới trong Titanic, trên thực tế có thể giải thích theo quy tắc sơ tan 1haan2g hải truyền thống "phụ nữ và trẻ em trước".

---
## (Bonus — không bắt buộc) Thử thách nâng cao

Chọn **một** trong các hướng sau nếu bạn muốn thử sức:

1. **Feature engineering:** tạo đặc trưng mới `family_size = sibsp + parch + 1`, hoặc trích `title` (Mr/Mrs/Miss...) từ tên (nếu dùng bản có cột `name`). Kiểm tra tương quan với `survived`.
2. **So sánh scaler:** vẽ phân phối `fare` trước và sau khi áp `StandardScaler`, `MinMaxScaler`, `RobustScaler`. Nhận xét scaler nào phù hợp nhất với dữ liệu lệch + có outlier.
3. **Bẫy KNN:** thử `KNNImputer` để điền `age` **khi chưa scale** và **sau khi đã scale** `fare`. Quan sát kết quả có khác nhau không, và giải thích tại sao (gợi ý: khoảng cách Euclid bị chi phối bởi cột thang đo lớn).

In [ ]:
# (tùy chọn) code cho phần Bonus
# Em chọn phần 2 nhe.
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

fare = df[["fare"]].dropna() # Chọn cột fare và bỏ giá trị thiếu
standard = StandardScaler().fit_transform(fare)
minmax = MinMaxScaler().fit_transform(fare)
robust = RobustScaler().fit_transform(fare)

# Vẽ biểu đồ
plt.figure(figsize=(6,4))
sns.histplot(fare["fare"], bins=20, kde=True)
plt.title("Fare một mình")
plt.show()

plt.figure(figsize=(6,4))
sns.histplot(standard.flatten(), bins=20, kde=True)
plt.title("Fare sau khi áp StandardScaler")
plt.show()

plt.figure(figsize=(6,4))
sns.histplot(minmax.flatten(), bins=20, kde=True)
plt.title("Fare sau khi áp MinMaxScaler")
plt.show()

plt.figure(figsize=(6,4))
sns.histplot(robust.flatten(), bins=20, kde=True)
plt.title("Fare sau khi áp RobustScaler")
plt.show()

Fare một mình có phân phối lệch sang phải và có khá nhiều outlier. Fare sau khi áp StandardScaler 1 phần vẫn còn outlier nhưng đã chuyển dữ liệu về trung bình 0 và độ lệch chuẩn 1. Fare sau khi áp MinMaxScaler đã đưa dữ liệu về khoảng từ 0 đến 1, nhưng vẫn bị ảnh hưởng của outlier, làm cho phần lớn dữ liệu lệch về phải - gần 0. Fare sau khi áp RobustScaler ít bị ảnh hưởng bởi outlier do sử dụng median và IQR => vì vậy sử dụng RobustScaler là lựa chọn phù hợp nhất.

---
## Bảng tự kiểm trước khi nộp

- [ ] Notebook chạy **Restart & Run All** không lỗi.
- [ ] Đã bỏ các cột rò rỉ/dư thừa (Task 1) và giải thích được vì sao.
- [ ] Mỗi biểu đồ (Task 5) đều có **nhận xét**.
- [ ] Đã **chia tập trước**, tiền xử lý **fit chỉ trên train** (Task 6–7).
- [ ] Đã trả lời tất cả các phần *"Trả lời:"*.
- [ ] Nhận xét tổng hợp (Task 9) có **ít nhất 5 ý** dựa trên bằng chứng.
- [ ] Đã push lên **repo cá nhân trên GitHub**.
